In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


##cust_info

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")


TABLE_PATH = 'workspace.bronze_new.crm_20260221135014660445_cust_info'

display(spark.sql('SHOW COLUMNS IN {0}'.format(TABLE_PATH)))

In [0]:
from pyspark.sql.functions import col, trim, lower, upper, substring, initcap, to_date, current_timestamp, when

TABLE_PATH = 'workspace.bronze_new.crm_20260221135014660445_cust_info'

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Check nulls for the Primary keys and drop duplicates
df_silver = (df
             .dropna(subset=['cst_id', 'cst_key']) # Safer to drop nulls on both keys
             .dropDuplicates(['cst_key'])
             )

# 3. Adjust names (Overwriting the existing columns to avoid clutter)
df_silver = (df_silver
             .withColumn('cst_firstname', initcap(trim(col('cst_firstname'))))
             .withColumn('cst_lastname', initcap(trim(col('cst_lastname'))))
             )

# 4. Standardize gender and marital status
df_silver = (df_silver
             .withColumn('cst_gndr', when(col('cst_gndr').isNotNull(), upper(substring(trim(col('cst_gndr')), 1, 1))).otherwise('Unknown')))


df_silver = (df_silver.withColumn('cst_marital_status', 
                         when(lower(trim(col('cst_marital_status'))).startswith('s'), 'Single')
                        .when(lower(trim(col('cst_marital_status'))).startswith('m'), 'Married')
                        .otherwise('Unknown')))

# 5. Parse dates and add metadata
df_silver = (df_silver
             .withColumn('cst_create_date', to_date(col('cst_create_date'), 'MM/dd/yyyy'))
             .withColumn("_cleaned_timestamp", current_timestamp())
             )

# 6. Write to Silver
(df_silver.write
 .format('delta') # Explicitly declaring delta is a good habit
 .mode('overwrite')
 .saveAsTable('workspace.silver.crm_cust_info')
 )

print(f"Successfully cleaned {TABLE_PATH} and wrote to the Silver layer!")

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.crm_20260221135014660445_cust_info LIMIT 10'))

In [0]:
display(spark.sql('SELECT * FROM workspace.silver.crm_cust_info LIMIT 10'))

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.crm_20260221135014660445_cust_info WHERE cst_id = 11173'))

##prd_info

In [0]:
TABLE_PATH = 'workspace.bronze_new.crm_20260221135017183310_prd_info'

display(spark.sql('SHOW COLUMNS IN {0}'.format(TABLE_PATH)))

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.crm_20260221135017183310_prd_info LIMIT 10'))

In [0]:
df = spark.read.table("workspace.bronze_new.crm_20260221135017183310_prd_info")

df.filter(col('prd_cost').isNull()).count()


In [0]:
from pyspark.sql.functions import *

TABLE_PATH = 'workspace.bronze_new.crm_20260221135017183310_prd_info'

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Check nulls for the Primary keys and drop duplicates
df_silver = (df
             .dropna(subset=['prd_id', 'prd_key']) # Safer to drop nulls on both keys
             .dropDuplicates(['prd_key'])
             )

# 3. Adjust names (Overwriting the existing columns to avoid clutter)
df_silver = (df_silver
             .withColumn('prd_nm', initcap(trim(col('prd_nm'))))
             .withColumn('prd_line', upper(trim(col('prd_line'))))
             )

# cast cost to numeric value.
df_silver = (df_silver
             .withColumn('prd_cost', col('prd_cost').cast('decimal(10,2)')))


# 6. Write to Silver
(df_silver.write
 .format('delta') # Explicitly declaring delta is a good habit
 .mode('overwrite')
 .saveAsTable('workspace.silver.prd_info')
 )

print(f"Successfully cleaned {TABLE_PATH} and wrote to the Silver layer!")

##sales

In [0]:
TABLE_PATH = 'workspace.bronze_new.crm_20260221135017680009_sales_details'

display(spark.sql('SHOW COLUMNS IN {0}'.format(TABLE_PATH)))

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.crm_20260221135017680009_sales_details LIMIT 20'))

In [0]:
from pyspark.sql.functions import col, to_date, regexp_replace , try_to_date

TABLE_PATH = 'workspace.bronze_new.crm_20260221135017680009_sales_details'

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Check nulls for the Primary keys and drop duplicates
df_silver = (df
             .dropna(subset=['sls_ord_num', 'sls_prd_key' , 'sls_cust_id'])
             .dropDuplicates(['sls_ord_num', 'sls_prd_key']))

# 3. Cast date to date value.
df_silver = (df_silver
             .withColumn('sls_order_dt', try_to_date(col('sls_order_dt').cast('string'), 'yyyyMMdd'))
             .withColumn('sls_ship_dt', try_to_date(col('sls_ship_dt').cast('string'), 'yyyyMMdd'))
             .withColumn('sls_due_dt', try_to_date(col('sls_due_dt').cast('string'), 'yyyyMMdd')))

# 4. Clean financial columns 
# FIX: Notice the 'r' right before the regex strings!
df_silver = (df_silver
             .withColumn('sls_sales', regexp_replace(col('sls_sales').cast('string'), r'[\$,]', '').cast('decimal(10,2)'))
             .withColumn('sls_price', regexp_replace(col('sls_price').cast('string'), r'[\$,]', '').cast('decimal(10,2)'))
             .withColumn('sls_quantity', col('sls_quantity').cast('int'))
             )

# 5. Write to Silver
(df_silver.write
 .format('delta') 
 .mode('overwrite')
 .saveAsTable('workspace.silver.sales_details')
 )

print(f"Successfully cleaned {TABLE_PATH} and wrote to the Silver layer!")

##cust_az12

In [0]:
TABLE_PATH = 'workspace.bronze_new.erp_20260221135020384837_cust_az12'

display(spark.sql('SHOW COLUMNS IN {0}'.format(TABLE_PATH)))

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.erp_20260221135020384837_cust_az12 LIMIT 10'))

In [0]:
from pyspark.sql.functions import col, trim, upper, substring, try_to_date, current_timestamp

TABLE_PATH = 'workspace.bronze_new.erp_20260221135020384837_cust_az12' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .dropna(subset=['CID'])
             .dropDuplicates(['CID'])
             )

# 3. Categorical Cleanup (Standardize Gender to a single uppercase letter)
df_silver = (df_silver
             .withColumn('GEN', upper(substring(trim(col('GEN')), 1, 1)))
             )

# 4. Date Parsing (Safely converting Birth Date)
# Using try_to_date to catch any weird default ERP values (like '0') and turn them into nulls
df_silver = (df_silver
             .withColumn('BDATE', try_to_date(col('BDATE').cast('string'), 'yyyyMMdd'))
             )

# 5. Metadata Injection
df_silver = df_silver.withColumn("_cleaned_timestamp", current_timestamp())

# 6. Write to Silver Layer
(df_silver.write
 .format('delta')
 .mode('overwrite')
 .saveAsTable('workspace.silver.erp_cust_az12')
 )

print(f"Successfully cleaned Customer Demographics and wrote to workspace.silver.erp_cust_az12!")

##loc_a101

In [0]:
TABLE_PATH = 'workspace.bronze_new.erp_20260221135021821657_loc_a101'

display(spark.sql('SHOW COLUMNS IN {0}'.format(TABLE_PATH)))

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.erp_20260221135021821657_loc_a101 LIMIT 10'))

In [0]:
from pyspark.sql.functions import col, trim, upper, substring, try_to_date, current_timestamp

TABLE_PATH = 'workspace.bronze_new.erp_20260221135021821657_loc_a101' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .dropna(subset=['CID'])
             .dropDuplicates(['CID'])
             )

# 3. Categorical Cleanup (Standardize Gender to a single uppercase letter)
df_silver = (df_silver
             .withColumn('CNTRY', upper(trim(col('CNTRY'))))
             .fillna({'CNTRY': 'UNKNOWN'}) # Replaces true nulls with a readable string
             )



# 6. Write to Silver Layer
(df_silver.write
 .format('delta')
 .mode('overwrite')
 .saveAsTable('workspace.silver.loc_a101')
 )

print(f"Successfully cleaned Customer Demographics and wrote to workspace.silver.loc_a101!")

##px_cat_g1v2

In [0]:
TABLE_PATH = 'workspace.bronze_new.erp_20260221135023079340_px_cat_g1v2'

display(spark.sql('SHOW COLUMNS IN {0}'.format(TABLE_PATH)))

In [0]:
display(spark.sql('SELECT * FROM workspace.bronze_new.erp_20260221135023079340_px_cat_g1v2 LIMIT 10'))

In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when

TABLE_PATH = 'workspace.bronze_new.erp_20260221135023079340_px_cat_g1v2' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .dropna(subset=['ID'])
             .dropDuplicates(['ID'])
             )

# 3. Categorical Cleanup (Standardize text for BI filtering)
df_silver = (df_silver
             .withColumn('CAT', upper(trim(col('CAT'))))
             .withColumn('SUBCAT', upper(trim(col('SUBCAT'))))
             )

# 4. Maintenance Flag Standardization
# Forces the text to uppercase, then maps common ERP codes to strict YES/NO
df_silver = (df_silver
             .withColumn('MAINTENANCE', upper(trim(col('MAINTENANCE'))))
             .withColumn('MAINTENANCE', 
                         when(col('MAINTENANCE').isin('Y', '1', 'TRUE', 'YES', 'REQ'), 'YES')
                        .when(col('MAINTENANCE').isin('N', '0', 'FALSE', 'NO', 'NONE'), 'NO')
                        .otherwise(col('MAINTENANCE'))) # Keeps the original text if it doesn't match above
             .fillna({'MAINTENANCE': 'UNKNOWN'}) # Fills actual nulls
             )


# 6. Write to Silver Layer
(df_silver.write
 .format('delta')
 .mode('overwrite')
 .saveAsTable('workspace.silver.erp_px_cat_g1v2')
 )

print("Successfully cleaned Category Mapping data and wrote to workspace.silver.erp_px_cat_g1v2!")

In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, regexp_replace

TABLE_PATH = 'workspace.bronze_new.erp_20260221135023079340_px_cat_g1v2' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .withColumn("ID",regexp_replace(col("ID"),"_", "-")
             ))
display(df_silver)


In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, regexp_replace , substring

TABLE_PATH = 'workspace.bronze_new.crm_20260221135017183310_prd_info' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .withColumn("cat_id",substring(col("prd_key"),1, 5)))
display(df_silver)


In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, regexp_replace, length, substring

TABLE_PATH = 'workspace.bronze_new.crm_20260221135017183310_prd_info' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .withColumn("sls_prd_key",substring(col("prd_key"),7,length(col("prd_key")))))
display(df_silver)


In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, regexp_replace, length, substring

TABLE_PATH = 'workspace.bronze_new.erp_20260221135021821657_loc_a101' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .withColumn("CID",regexp_replace(col("CID"),"-","")))
display(df_silver)


In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, regexp_replace, length, substring

TABLE_PATH = 'workspace.bronze_new.erp_20260221135020384837_cust_az12' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .withColumn("CID", substring(col("CID"), 4, length(col("CID")))))


display(df_silver)


In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, regexp_replace, length, substring

TABLE_PATH = 'workspace.bronze_new.crm_20260221135017680009_sales_details' # Update to your exact bronze table name

# 1. Read the data
df = spark.read.table(TABLE_PATH)

# 2. Strict Key Cleanup (Deduplication)
df_silver = (df
             .withColumn("sls_sales", substring(col("CID"), 4, length(col("CID")))))


display(df_silver)
